# Feature Stores and Reuse

When several models use the same features ("trips in the last hour in this zone", "customer's average basket"), each team tends to recompute them slightly differently, and the values used in training differ from those computed in production. A **feature store** defines each feature once, stores its history for training (offline store) and its latest value for real-time predictions (online store).

**After this lesson you can:**

1. Explain entities, feature views, event timestamps, offline and online stores.
2. Build **point-in-time correct** training data with an as-of join.
3. Serve the latest feature values from an online store (Redis).
4. Check training/serving consistency, and know when a tool like Feast is worth it.

**Data:** NYC yellow taxi trips, January–February 2024. Entity: the pickup zone. Online store: Redis (Docker, profile `db`).

In [1]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

sys.path.insert(0, str(Path.cwd().parent / "tools"))
import datasets                                   # course helper: downloads pinned public datasets

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})
RNG = 42

import json
import tempfile

import polars as pl
import redis

STORE = Path(tempfile.mkdtemp())                                       # the "offline store": Parquet files
datasets.ensure_service("db", "redis")
r = redis.Redis(host="localhost", port=6379, decode_responses=True)
print("Redis ping:", r.ping())

Redis ping: True


## 1. Concepts

| Concept | Meaning | Example |
| :--- | :--- | :--- |
| **entity** | the object features describe; has a key | pickup zone (`zone_id`) |
| **feature view** | a group of features computed by one definition | hourly zone demand features |
| **event timestamp** | when a feature value became known | end of the hour it summarises |
| **offline store** | full history, for building training sets | Parquet, a warehouse |
| **online store** | latest value per entity, low latency | Redis, DynamoDB |
| **point-in-time join** | for each training example, use the feature values known at that moment | `merge_asof` |

## 2. Define and Materialise a Feature View

One function defines the features; its output is written to the offline store with an `event_timestamp` = end of the hour (when the value becomes known).

In [2]:
FEATURE_VIEW = {
    "name": "zone_hourly_demand",
    "entity": "zone_id",
    "features": ["trips_last_hour", "median_fare_last_hour", "share_card_last_hour"],
    "ttl_hours": 3,                                  # values older than this are treated as missing
    "version": 1,
}

def compute_zone_hourly(month: str) -> pd.DataFrame:
    """The single definition of the feature view (used for history AND for fresh values)."""
    start = pl.datetime(int(month[:4]), int(month[5:]), 1)
    df = (pl.scan_parquet(datasets.taxi(month))
            .filter(pl.col("tpep_pickup_datetime") >= start, pl.col("tpep_pickup_datetime") < start.dt.offset_by("1mo"),
                    pl.col("fare_amount").is_between(3, 500))
            .group_by(pl.col("PULocationID").alias("zone_id"), pl.col("tpep_pickup_datetime").dt.truncate("1h").alias("hour"))
            .agg(trips_last_hour=pl.len(), median_fare_last_hour=pl.col("fare_amount").median(),
                 share_card_last_hour=(pl.col("payment_type") == 1).mean())
            .with_columns(event_timestamp=pl.col("hour") + pl.duration(hours=1), zone_id=pl.col("zone_id").cast(pl.Int64))
            .drop("hour").sort("zone_id", "event_timestamp").collect())
    return df.to_pandas()

offline = pd.concat([compute_zone_hourly("2024-01"), compute_zone_hourly("2024-02")], ignore_index=True)
offline.to_parquet(STORE / f"{FEATURE_VIEW['name']}_v{FEATURE_VIEW['version']}.parquet", index=False)
print(f"offline store: {len(offline):,} rows for {offline['zone_id'].nunique()} zones")
offline.head(3)

offline store: 151,635 rows for 261 zones


,zone_id,trips_last_hour,median_fare_last_hour,share_card_last_hour,event_timestamp
0,1,1,90.0,1.000000,2024-01-01 06:00:00
1,1,3,85.0,0.666667,2024-01-01 07:00:00
2,1,3,80.0,1.000000,2024-01-01 14:00:00


## 3. Point-in-Time Correct Training Data

Training examples ("entity rows") have a key and a timestamp: here, trips with their pickup zone and pickup time; the label is the trip's tip percentage. For each row, the join must use the latest feature row with `event_timestamp <= pickup time`, within the TTL. A naive join on "the same hour" would use the full hour, including minutes **after** the pickup: leakage.

In [3]:
trips = pd.read_parquet(datasets.taxi("2024-02"), columns=["tpep_pickup_datetime", "PULocationID", "fare_amount", "tip_amount", "payment_type"])
trips = trips[(trips["payment_type"] == 1) & trips["fare_amount"].between(3, 500)
              & trips["tpep_pickup_datetime"].between("2024-02-01", "2024-03-01", inclusive="left")]
entity_rows = (trips.sample(50_000, random_state=RNG)
               .rename(columns={"PULocationID": "zone_id", "tpep_pickup_datetime": "ts"}).astype({"zone_id": "int64"})
               .assign(tip_pct=lambda d: 100 * d["tip_amount"] / d["fare_amount"])[["zone_id", "ts", "tip_pct"]]
               .sort_values("ts"))

def get_historical_features(entity_rows: pd.DataFrame, view: dict) -> pd.DataFrame:
    feats = pd.read_parquet(STORE / f"{view['name']}_v{view['version']}.parquet").sort_values("event_timestamp")
    out = pd.merge_asof(entity_rows.sort_values("ts"), feats, left_on="ts", right_on="event_timestamp", by=view["entity"],
                        direction="backward", tolerance=pd.Timedelta(hours=view["ttl_hours"]))
    return out

train = get_historical_features(entity_rows, FEATURE_VIEW)
assert (train["event_timestamp"].dropna() <= train.loc[train["event_timestamp"].notna(), "ts"]).all()
print(f"training rows: {len(train):,}; rows without a fresh feature value (TTL): {train['trips_last_hour'].isna().mean():.1%}")
train.head(3)

training rows: 50,000; rows without a fresh feature value (TTL): 0.3%


,zone_id,ts,tip_pct,trips_last_hour,median_fare_last_hour,share_card_last_hour,event_timestamp
0,211,2024-02-01 00:00:16,26.711409,36.0,11.4,0.861111,2024-02-01
1,113,2024-02-01 00:01:44,37.075472,61.0,11.4,0.950820,2024-02-01
2,161,2024-02-01 00:01:56,28.264463,119.0,11.4,0.831933,2024-02-01


In [4]:
naive = entity_rows.assign(hour=entity_rows["ts"].dt.floor("h") + pd.Timedelta(hours=1)).merge(
    offline, left_on=["zone_id", "hour"], right_on=["zone_id", "event_timestamp"], how="left")
future = (naive["event_timestamp"] > naive["ts"]).mean()
print(f"naive same-hour join: {future:.0%} of rows use a feature value that was not yet known at pickup time")

naive same-hour join: 100% of rows use a feature value that was not yet known at pickup time


## 4. Online Store: Serve the Latest Values

For real-time predictions, the model needs the **latest** values per zone within milliseconds. Materialisation copies them from the offline store to Redis (one hash per entity, with the event timestamp).

In [5]:
def materialize_online(view: dict, as_of: pd.Timestamp) -> int:
    feats = pd.read_parquet(STORE / f"{view['name']}_v{view['version']}.parquet")
    latest = feats[feats["event_timestamp"] <= as_of].sort_values("event_timestamp").groupby(view["entity"]).tail(1)
    with r.pipeline() as p:
        for row in latest.itertuples(index=False):
            key = f"fs:{view['name']}:v{view['version']}:{getattr(row, view['entity'])}"
            p.hset(key, mapping={f: float(getattr(row, f)) for f in view["features"]} | {"event_timestamp": row.event_timestamp.isoformat()})
        p.execute()
    return len(latest)

def get_online_features(view: dict, zone_ids: list[int], now: pd.Timestamp) -> pd.DataFrame:
    rows = []
    for z in zone_ids:
        h = r.hgetall(f"fs:{view['name']}:v{view['version']}:{z}")
        fresh = h and (now - pd.Timestamp(h["event_timestamp"])) <= pd.Timedelta(hours=view["ttl_hours"])
        rows.append({"zone_id": z, **({f: float(h[f]) for f in view["features"]} if fresh else {f: np.nan for f in view["features"]})})
    return pd.DataFrame(rows)

now = pd.Timestamp("2024-02-15 18:30")
n = materialize_online(FEATURE_VIEW, as_of=now)
t = time.perf_counter()
online = get_online_features(FEATURE_VIEW, [132, 138, 161, 237, 1], now)
latency = (time.perf_counter() - t) * 1000
print(f"materialised {n} zones; online lookup of 5 zones in {latency:.1f} ms")
online

materialised 261 zones; online lookup of 5 zones in 2.3 ms


,zone_id,trips_last_hour,median_fare_last_hour,share_card_last_hour
0,132,219.0,70.0,0.730594
1,138,173.0,44.3,0.843931
2,161,622.0,13.5,0.800643
3,237,455.0,10.7,0.819780
4,1,2.0,69.5,1.000000


## 5. Training/Serving Consistency Check

Because both stores come from **one** definition, the online value for a zone at time *t* must equal the offline (point-in-time) value for the same zone and time. Checking this automatically catches skew between the training and serving paths.

In [6]:
check_rows = pd.DataFrame({"zone_id": np.array([132, 138, 161, 237], dtype="int64"), "ts": [now] * 4, "tip_pct": np.nan})
offline_view = get_historical_features(check_rows, FEATURE_VIEW).set_index("zone_id")[FEATURE_VIEW["features"]]
online_view = get_online_features(FEATURE_VIEW, [132, 138, 161, 237], now).set_index("zone_id")[FEATURE_VIEW["features"]]
consistent = np.allclose(offline_view.to_numpy(), online_view.to_numpy(), equal_nan=True)
print("offline and online values identical:", consistent)
print(pd.concat({"offline": offline_view, "online": online_view}, axis=1).round(3).head(2))

offline and online values identical: True
                offline                                             \
        trips_last_hour median_fare_last_hour share_card_last_hour   
zone_id                                                              
132                 219                  70.0                0.731   
138                 173                  44.3                0.844   

                 online                                             
        trips_last_hour median_fare_last_hour share_card_last_hour  
zone_id                                                             
132               219.0                  70.0                0.731  
138               173.0                  44.3                0.844  


## 6. When to Use a Feature Store Tool

The same concepts in **Feast** (open source) look like this:

```python
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64
from datetime import timedelta

zone = Entity(name="zone", join_keys=["zone_id"])
source = FileSource(path="zone_hourly_demand_v1.parquet", timestamp_field="event_timestamp")
zone_hourly_demand = FeatureView(
    name="zone_hourly_demand", entities=[zone], ttl=timedelta(hours=3), source=source,
    schema=[Field(name="trips_last_hour", dtype=Int64), Field(name="median_fare_last_hour", dtype=Float32),
            Field(name="share_card_last_hour", dtype=Float32)])
# store.get_historical_features(entity_df, features=[...])   -> point-in-time training data
# store.materialize(start, end); store.get_online_features(...)   -> online serving
```

| Situation | Recommendation |
| :--- | :--- |
| one model, batch predictions | a shared feature function and Parquet tables are enough |
| several models share features, real-time predictions | a feature store (Feast, Tecton, Databricks/Vertex/SageMaker feature stores) |
| strict point-in-time requirements (finance, fraud) | a feature store or carefully tested as-of joins |

*Insight: The core of a feature store is simple: one definition per feature, event timestamps, point-in-time joins for training, and the same values materialised for serving. Adopt a tool when several models and real-time serving make these guarantees hard to maintain by hand.*

## Real-World Use Case or Analogy

A feature store is like a **central kitchen** for a restaurant chain: sauces are prepared once with one recipe (definition), stored with a date label (event timestamp), and delivered fresh to every restaurant (online store). Without it, each restaurant makes its own version, and the dish tastes different everywhere.

Ride-hailing, food delivery and payment companies built the first feature stores (for example Uber's Michelangelo) because real-time models for prices, ETAs and fraud need exactly these guarantees.